# 10 - Due Diligence Agent
## Multi-Document Risk Extraction + Severity x Likelihood Matrix
25 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/10-due-diligence/due_diligence_workbook.ipynb)

A real due diligence process means reading dozens of documents -- management accounts,
contracts, director bios, regulatory correspondence -- and synthesising them into a
single risk register before a deal committee meeting.

This example builds a two-stage pipeline:
1. **Extractor** -- reads each document independently, pulls concrete findings
2. **Synthesiser** -- consolidates all findings into one DDReport with a 2D risk matrix

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | Why multi-document analysis needs two stages |
| 2 | Schema: DocumentFindings + RiskItem + DDReport |
| 3 | Stage 1: per-document extraction |
| 4 | Stage 2: synthesis into unified risk register |
| 5 | Run the full pipeline on four documents |
| * | Exercises + Answer Key |

### Prerequisites
- Python 3.10+, or Google Colab
- OPENAI_API_KEY in .env or Colab Secrets

In [ ]:
try:
    from google.colab import userdata  # noqa: F401
    import subprocess
    subprocess.run(["pip", "install", "-q", "langchain-openai", "pydantic", "python-dotenv"], check=True)
    print("Packages installed.")
except ImportError:
    print("Local environment.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Set OPENAI_API_KEY in .env or Colab Secrets.")
print("API key ready.")

---

## Part 1 - Why Two Stages?

If you feed all documents to a single LLM call, you get findings that blend evidence
from multiple sources without clear attribution.

The two-stage pattern separates concerns:

| Stage | Input | Output | Job |
|-------|-------|--------|-----|
| Extractor | One document | DocumentFindings | What does THIS document say? |
| Synthesiser | All DocumentFindings | DDReport | What does everything together mean? |

The extractor is fast and cheap. The synthesiser gets clean structured inputs
instead of a wall of raw text.

---

## Part 2 - Schema

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field


class DocumentFindings(BaseModel):
    document_name: str
    document_type: Literal[
        "financials", "contract", "management_bio", "regulatory", "corporate", "other"
    ]
    key_findings: List[str] = Field(
        description="Concrete facts from this document, each as one sentence"
    )
    red_flags: List[str] = Field(description="Issues that warrant closer scrutiny")
    questions_raised: List[str] = Field(description="Questions this document raises")


class RiskItem(BaseModel):
    area: Literal["financial", "commercial", "operational", "legal", "management", "regulatory"]
    severity: Literal["critical", "high", "medium", "low"]
    likelihood: Literal["high", "medium", "low"]
    title: str
    finding: str = Field(description="Evidence from the documents supporting this risk")
    source_document: str = Field(description="Which document(s) this came from")
    mitigation: str = Field(description="Recommended mitigation or further investigation")


class DDReport(BaseModel):
    target_company: Optional[str] = None
    documents_reviewed: List[str]
    overall_assessment: Literal["proceed", "proceed_with_conditions", "do_not_proceed"]
    executive_summary: str = Field(description="3-4 sentence summary for a deal committee")
    risk_items: List[RiskItem] = Field(description="Risks sorted by severity then likelihood")
    key_conditions: List[str] = Field(description="Conditions that must be met before proceeding")
    further_investigation: List[str] = Field(description="Areas needing deeper diligence")

print("Schema defined. Risk scoring: severity x likelihood (2D matrix)")

---

## Part 3 - Stage 1: Per-Document Extraction

In [ ]:
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI

EXTRACTOR_SYSTEM = SystemMessage(
    """You are a due diligence analyst extracting structured findings from a single document.
Be specific and factual. Every key_finding must be a concrete, verifiable statement
from the document. Every red_flag must cite specific evidence."""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
extractor = EXTRACTOR_SYSTEM | llm.with_structured_output(DocumentFindings)

MANAGEMENT_ACCOUNTS = """
ACME TECHNOLOGIES LTD -- MANAGEMENT ACCOUNTS FY2024 (UNAUDITED)
Revenue: GBP 4.2m (+35% YoY) | Gross margin: 61% | EBITDA: GBP 0.3m
Net cash: GBP 0.1m | Monthly burn: GBP 180k

Revenue: RetailCo 50%, LogisCorp 19%, Other 31%
Recurring SaaS: 68% | Professional services: 32%

Three senior engineers resigned Q4 2024. Debtor days: 87 (benchmark: 45).
Two RetailCo invoices (GBP 310k) overdue 90+ days.
"""

findings = extractor.invoke(
    f"Document name: Management Accounts (FY2024)\n\nDocument text:\n{MANAGEMENT_ACCOUNTS}"
)
print(f"Document: {findings.document_name} | Type: {findings.document_type}")
print(f"\nKey findings ({len(findings.key_findings)}):")
for f in findings.key_findings:
    print(f"  - {f}")
print(f"\nRed flags ({len(findings.red_flags)}):")
for r in findings.red_flags:
    print(f"  ! {r}")

---

## Part 4 - Stage 2: Synthesis

In [ ]:
SYNTHESISER_SYSTEM = SystemMessage(
    """You are a senior M&A advisor synthesising due diligence document reviews
into a unified risk register for a deal committee.

- Consolidate overlapping findings into single risk items
- Score each risk on BOTH severity AND likelihood
- Source every risk to the document(s) it came from
- Make a clear overall_assessment
Quality over quantity."""
)

synthesiser = SYNTHESISER_SYSTEM | llm.with_structured_output(DDReport)

def synthesise(all_findings):
    text = "\n\n".join(
        "=== " + f.document_name + " (" + f.document_type + ") ===\n"
        "Key findings:\n" + "\n".join("- " + x for x in f.key_findings) + "\n"
        "Red flags:\n" + "\n".join("- " + x for x in f.red_flags)
        for f in all_findings
    )
    return synthesiser.invoke(
        "Synthesise into a unified DD report:\n\n" + text
    )

test_report = synthesise([findings])
print(f"Assessment: {test_report.overall_assessment}")
print(f"Risks: {len(test_report.risk_items)}")

---

## Part 5 - Full Pipeline: Four Documents

In [ ]:
KEY_CONTRACT = """
MASTER SERVICES AGREEMENT -- ACME TECHNOLOGIES LTD AND RETAILCO PLC
Term: expires 28 February 2025. No auto-renewal. ACV: GBP 2.1m.
Section 4: RetailCo may terminate with 30 days notice, no fee.
Section 7: All custom development becomes RetailCo property, including core platform modifications.
Section 9: Three consecutive SLA breaches entitle RetailCo to terminate without penalty.
Section 11: RetailCo may benchmark pricing annually and renegotiate if >15% above market.
"""

CEO_BIO = """
JAMES WHITFIELD -- CO-FOUNDER AND CEO
Owns 45%. Co-founder Sarah Chen (CTO) owns 40%. Angels 15%.
Prior venture DataFlow Ltd entered administration 2017. Creditors received 12p in pound.
Relocating to Singapore Q3 2025. Remote working arrangement agreed for 12 months.
"""

ICO_NOTICE = """
ICO ENFORCEMENT NOTICE -- 14 November 2024
Acme Technologies Ltd in breach of Article 32 UK GDPR. Data incident August 2024:
12,000 RetailCo end-user records exposed due to misconfigured database.
Requirements: security programme by 31 Mar 2025, DPO by 28 Feb 2025, remediation report by 30 Apr 2025.
Potential penalty: up to GBP 17.5m or 4% annual global turnover.
"""

documents = {
    "Management Accounts (FY2024)": MANAGEMENT_ACCOUNTS,
    "Key Customer Contract (RetailCo)": KEY_CONTRACT,
    "CEO Biography": CEO_BIO,
    "ICO Enforcement Notice": ICO_NOTICE,
}

all_findings = []
for doc_name, doc_text in documents.items():
    f = extractor.invoke(f"Document name: {doc_name}\n\nDocument text:\n{doc_text}")
    all_findings.append(f)
    print(f"Extracted {doc_name}: {len(f.key_findings)} findings, {len(f.red_flags)} flags")

report = synthesise(all_findings)

SEV = {"critical": "CRIT", "high": "HIGH", "medium": "MED ", "low": "LOW "}
LIK = {"high": "H", "medium": "M", "low": "L"}

print(f"\nASSESSMENT: {report.overall_assessment.upper()}")
print(f"\n{report.executive_summary}")
print(f"\nRISK REGISTER ({len(report.risk_items)} items):")
for r in report.risk_items:
    print(f"  [{SEV[r.severity]}][{LIK[r.likelihood]}] {r.area:<14} {r.title}")
    print(f"    Source: {r.source_document}")
if report.key_conditions:
    print("\nCONDITIONS:")
    for c in report.key_conditions:
        print(f"  * {c}")

---

## Exercise 1 - Add Valuation Impact Assessment

For each critical or high risk, estimate the deal impact:
- `price_chip`: negotiate a lower price
- `escrow`: hold part of consideration in escrow
- `warranty`: require W&I insurance cover
- `walk_away`: fatal to the deal

Add a `ValuationImpact` model and a `DDReportWithValuation` class that includes it.

In [ ]:
# Exercise 1 Starter
class ValuationImpact(BaseModel):
    risk_title: str
    impact_type: Literal["price_chip", "escrow", "warranty", "walk_away"]
    estimated_impact: Optional[str] = None
    rationale: str

# TODO: Create DDReportWithValuation with valuation_impacts: List[ValuationImpact]
# TODO: Update synthesiser system prompt to assess valuation impact for critical/high risks
# TODO: Run and print valuation_impacts

In [ ]:
# Exercise 1 Answer Key
class DDReportWithValuation(BaseModel):
    target_company: Optional[str] = None
    documents_reviewed: List[str]
    overall_assessment: Literal["proceed", "proceed_with_conditions", "do_not_proceed"]
    executive_summary: str
    risk_items: List[RiskItem]
    key_conditions: List[str]
    further_investigation: List[str]
    valuation_impacts: List[ValuationImpact] = Field(
        default_factory=list,
        description="Deal adjustments implied by critical and high risks"
    )

VALUATION_SYSTEM = SystemMessage(
    """You are a senior M&A advisor synthesising due diligence findings.
Also assess valuation impact for each critical or high severity risk:
- price_chip: negotiate lower price | escrow: hold back consideration
- warranty: require W&I cover | walk_away: fatal to deal
Only include valuation_impacts for critical and high severity risks."""
)

val_synthesiser = VALUATION_SYSTEM | llm.with_structured_output(DDReportWithValuation)
text = "\n\n".join(
    "=== " + f.document_name + " ===\n" + "\n".join("- " + x for x in f.key_findings + f.red_flags)
    for f in all_findings
)
report_v2 = val_synthesiser.invoke("Synthesise into a unified DD report:\n\n" + text)

print(f"Valuation impacts ({len(report_v2.valuation_impacts)}):")
for v in report_v2.valuation_impacts:
    print(f"  [{v.impact_type.upper()}] {v.risk_title}")
    if v.estimated_impact:
        print(f"    Impact: {v.estimated_impact}")
    print(f"    Why: {v.rationale}")

---

## What You Built

A two-stage due diligence pipeline demonstrating multi-document analysis:

1. Each document is extracted independently -- findings traceable to their source
2. A synthesiser consolidates all findings into a clean 2D risk matrix
3. `overall_assessment` gives the deal committee a clear go/no-go signal

The pattern scales: add more documents, add more extraction calls. The synthesiser
always receives clean structured inputs regardless of how many documents you process.

---
*Example 10 of 23 - agent-use-cases*